In [ ]:
import catboost
from catboost import CatBoostClassifier
from catboost.utils import get_roc_curve
# 获取类别特征索引（假设X_train是DataFrame）
cat_features_idx = [i for i, col in enumerate(X_train.columns) 
                    if X_train[col].dtype == 'object']  
# 创建并训练一个CatBoost分类器
model = CatBoostClassifier(iterations=1000,
                           task_type='GPU',
                           cat_features=feature_list,
                           eval_metric='AUC',
                           logging_level='Verbose',
                           learning_rate=0.05,
                           depth=6, 
                           l2_leaf_reg=5,
                           loss_function='Logloss',
                            early_stopping_rounds=300,
                           scale_pos_weight=(len(y_train[y_train==0])/len(y_train[y_train == 1])))
model.fit(X_train, y_train)
# 假设 df 是你的 Pandas DataFrame
train_data = catboost.Pool(data=X_train, label=y_train)
# 获取特征重要性评估
feature_importance = model.get_feature_importance(data=train_data, type='LossFunctionChange')
# 获取特征名称
feature_names = train_data.get_feature_names()
# 将特征重要性和特征名称结合起来，创建一个字典
feature_importance_dict = dict(zip(feature_names, feature_importance))

# 将特征重要性排序
sorted_feature_importance = sorted(feature_importance_dict.items(), key=lambda x: x[1], reverse=True)

# 提取排序后的特征名称和重要性
sorted_feature_names, sorted_feature_importance = zip(*sorted_feature_importance)

plt.figure(figsize=(10, 6))
plt.barh(sorted_feature_names[:10], sorted_feature_importance[:10])
plt.xlabel('Feature Importance')
plt.title('Top 10 Important Features')
plt.gca().invert_yaxis()  # 反转y轴以显示重要性高的特征在顶部
plt.show()
# 打印特征重要性得分
for i, score in enumerate(feature_importance):
    print(f'Feature {i}: {score}')